<a href="https://colab.research.google.com/github/Lucas-O-S/CalculadoraMedia/blob/main/AdrianaTeste" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Código para análise de intervalos estatísticos

Bibliotecas

In [13]:
import math
import pandas as pd



Função para:
1. Contar o número de dados e organizar-los em ordem crescente(do menor para o maior), não cria outra lista, ela altera a própria lista
2. Calcula o número de classes segundo o número de dados, arredonda pra cima.
3. Calcula amplitude total (distância entre o maior número e o menor)
4. Calcula amplitude das classes, arredonda pra cima. É o intervalo que cada classe deve compreender
5. Cria uma lista com k intervalos de h dos dados
6. Conta quantos dados caem em cada classe (fi) e também calcula o ponto médio (Xm) de cada intervalo
7. Calculo da média agrupada
8. Frequencia acumulada que é utilizada para encontrar a mediana, quartis e a distribuição acumulada dos dados
9. Mediana: encontra pela semelhança de triangulos, isolando a mediana para criar uma formula.
10. Variança, desvio padrão e coeficiente de variação

In [ ]:
def calcular_estatisticas(dados):
    # 1. Obter número de elementos e ordenar os dados
    n = len(dados)
    dados = dados.sort_values().reset_index(drop=True)  # retorna nova série ordenada
    
    # 2. Número de classes (regra de Sturges simples: ceil(sqrt(n)))
    k = math.ceil(math.sqrt(n))
    
    # 3. Amplitude total
    H = max(dados) - min(dados) if n > 0 else 0
    
    # 4. Amplitude de cada classe (arredonda para cima)
    h = math.ceil(H / k) if k > 0 else 0
    
    # 5. Construir os intervalos de classe
    classes = []
    inicio = min(dados) if n > 0 else 0
    for _ in range(k):
        fim = inicio + h
        classes.append((inicio, fim))
        inicio = fim
    
    # 6. Calcular fi e ponto médio para cada classe
    frequencias = []
    pontos_medios = []
    for i, (li, ls) in enumerate(classes):
        if i == len(classes) - 1:
            # última classe inclui o limite superior
            fi = sum(1 for x in dados if li <= x <= ls)
        else:
            fi = sum(1 for x in dados if li <= x < ls)
        frequencias.append(fi)
        pontos_medios.append((li + ls) / 2)
    # note: não é necessário ajuste adicional da última classe; a contagem já inclui o valor máximo
    
    # 7. Média agrupada
    soma = sum(pm * f for pm, f in zip(pontos_medios, frequencias))
    media = soma / n if n > 0 else 0
    
    # 8. Frequência acumulada
    fa = []
    acumulado = 0
    for f in frequencias:
        acumulado += f
        fa.append(acumulado)
    
    # 9. Mediana (dados agrupados)
    mediana = None
    if n > 0 and fa:
        pos = n / 2
        for j, f_ac in enumerate(fa):
            if f_ac >= pos:
                classe_mediana = classes[j]
                F_anterior = fa[j-1] if j > 0 else 0
                fi_cl = frequencias[j]
                L = classe_mediana[0]
                mediana = L + ((pos - F_anterior) / fi_cl) * h
                break
    
    # Medidas de dispersão
    variancia = sum(f * (pm - media) ** 2 for pm, f in zip(pontos_medios, frequencias)) / n if n > 0 else 0
    desvio = math.sqrt(variancia)
    cv = (desvio / media) * 100 if media != 0 else float('nan')
    
    # Montar tabela de saída
    tabela = pd.DataFrame({
        "Classe": classes,
        "fi": frequencias,
        "Xm": pontos_medios,
        "fa": fa
    })
    
    return {
        "tabela": tabela,
        "media": media,
        "mediana": mediana,
        "variancia": variancia,
        "desvio_padrao": desvio,
        "cv": cv
    }

Como os dados vão ser pegos? se for pra digitar no código precisa criar uma lista dados[] e colocar os valores separados por virgula

# Resultado

In [42]:
data = pd.read_excel("Data.xlsx", header=None)

data = pd.Series(data.values.flatten())

resultado = calcular_estatisticas(data)
print("Tabela de Frequência:\n")
print(resultado["tabela"])

print("\nMédia:", resultado["media"])
print("Mediana:", resultado["mediana"])
print("Variância:", resultado["variancia"])
print("Desvio padrão:", resultado["desvio_padrao"])
print("Coeficiente de Variação (%):", resultado["cv"])

Tabela de Frequência:

     Classe  fi    Xm  fa
0  (45, 52)   5  48.5   5
1  (52, 59)  10  55.5  15
2  (59, 66)   9  62.5  24
3  (66, 73)   8  69.5  32
4  (73, 80)   4  76.5  36
5  (80, 87)   3  83.5  39
6  (87, 94)   2  90.5  41

Média: 66.3375
Mediana: 62.888888888888886
Variância: 136.05894140625003
Desvio padrão: 11.664430607888669
Coeficiente de Variação (%): 17.583464266649585
